In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l8.5 Paged attention

> 🎯 **Goal:** Understand how fixed-size pages eliminate the wasted memory of a contiguous KV cache, using a tiny arithmetic demo of the page bookkeeping.

**Why this matters.** From the memory lesson you know the KV cache is the term that runs you out of GPU. A naive cache makes it worse: to hold a sequence that *might* grow to the max length, you reserve the full max length up front for *every* request, even the short ones. Most of that reservation sits empty. Paged attention is the idea (introduced by vLLM) that recovered huge amounts of that wasted memory and let one GPU serve far more concurrent users.

**The intuition.** This is the operating-system virtual-memory trick applied to the KV cache. An OS does not hand a program one giant contiguous block of RAM, it hands out small fixed-size pages on demand and keeps a table mapping logical addresses to physical pages. Paged attention does the same for the cache: store it in small fixed-size pages, allocate a new page only when the current one fills, and keep a table mapping each sequence to its pages. Two sequences that share a common prompt prefix can even point at the *same* physical pages for that prefix, like two processes sharing a read-only library in memory.

**The mechanics.** This lesson is a concept demo at pocket scale: we do not run a real GPU kernel or vLLM here, we just do the page arithmetic so the bookkeeping is concrete. Pick a `page_size` and a `seq_len`. The number of pages is the ceiling of `seq_len / page_size`. The slots actually used is `seq_len`; the slots reserved is `n_pages * page_size`. The difference is the wasted memory, and the key result is that this waste is at most one partial page per sequence, no matter how long the sequence is. Contrast that with a contiguous cache, where the waste is (max_length minus actual_length) for every request.

In [ ]:
page_size = 4
seq_len = 10
n_pages = (seq_len + page_size - 1) // page_size      # ceil division
used = seq_len
reserved = n_pages * page_size
print(f"seq {seq_len} -> {n_pages} pages of {page_size}")
print(f"slots used {used}, reserved {reserved}, wasted {reserved - used}")

**What just happened.** A 10-token sequence needed 3 pages of size 4, reserving 12 slots to hold 10, wasting 2. That waste is bounded by one page (here, less than 4 slots) regardless of how long the sequence grows, instead of reserving the full max length up front. Combined with sharing pages across requests that share a prompt prefix, this is how a paged cache fits many more concurrent users on a single GPU. Remember this is the arithmetic of the idea, not a running vLLM kernel.

⚠️ **Common pitfalls**
- Confusing this demo with a real implementation. Production paged attention needs custom CUDA kernels that gather K and V across non-contiguous pages, which we are not running here.
- Picking a tiny page size to cut waste. Smaller pages mean more pages and more page-table overhead, so there is a sweet spot, not a free lunch.
- Forgetting that prefix sharing requires the shared pages to be read-only, so a copy-on-write step is needed the moment two sequences diverge.

🏋️ **Try it yourself.** Change `seq_len` to 12 (an exact multiple of `page_size`) and see the waste drop to zero, then to 13 and watch a whole fresh page get reserved for a single extra token. Vary `page_size` between 1 and 16 and tabulate the wasted slots to feel the trade-off.

In [ ]:
assert n_pages == 3
assert reserved - used < page_size      # at most one partial page wasted